# EDA Stratégique — Dataset Handover 6G (ns-3)
## Analyse Exploratoire pour Débloquer le Plateau à 61% Top-1 Accuracy

> **Contexte métier :** Notre modèle DeepSet/Transformer atteint 61% en Top-1 et 89% en Top-3.  
> L'écart de 28 pp révèle une **ambiguïté décisionnelle** : le modèle identifie les bons candidats mais hésite entre eux.  
> Ce notebook est conçu pour diagnostiquer trois verrous :  
> 1. **Le Temporal Alias (TTT)** : le label est décalé dans le temps par rapport à la réalité physique  
> 2. **L'Ambiguïté de Voisinage** : plusieurs cellules ont des RSRP quasi-identiques  
> 3. **Le Déséquilibre de Classes** : `no_handover` domine massivement  

---
**Dataset :** Simulateur ns-3 — Handover 6G  
**Auteur :** Expert Senior Data Science / Réseaux Mobiles  
**Date :** Avril 2026


---
## Section 0 — Setup & Imports


In [17]:
# ── Imports ────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
import warnings

warnings.filterwarnings("ignore")

# ── Style global ───────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 130,
    "figure.facecolor": "#0f1117",
    "axes.facecolor": "#1a1d27",
    "axes.edgecolor": "#3d4166",
    "axes.labelcolor": "#e0e0f0",
    "axes.titlecolor": "#c8c8f0",
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.color": "#a0a0c0",
    "ytick.color": "#a0a0c0",
    "text.color": "#e0e0f0",
    "grid.color": "#2a2d3d",
    "grid.linestyle": "--",
    "grid.alpha": 0.6,
    "legend.facecolor": "#1a1d27",
    "legend.edgecolor": "#3d4166",
    "legend.labelcolor": "#e0e0f0",
    "font.family": "DejaVu Sans",
})

PALETTE_CLASS = {
    "no_handover":      "#4a9eff",
    "intra_freq_ho":    "#ff7043",
    "inter_rat_ho":     "#66bb6a",
    "drone_to_macro_ho":"#ab47bc",
    "inter_freq_ho":    "#ffa726",
}

print("✅ Imports OK")


✅ Imports OK


---
## Section 1 — Chargement, Nettoyage & Feature Engineering

### Pourquoi cette section est cruciale
Le dataset ns-3 contient des colonnes à valeurs multiples (séparateur `;`) encodant les **K voisins**
de chaque UE à chaque timestamp. Une extraction correcte de ces listes est le fondement de toute
l'analyse suivante, notamment pour calculer la **marge de hystérésis** et le **Δ RSRP**.


In [18]:
import numpy  as np
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing  import List, Tuple, Dict, Optional


_ROOT = Path("../../").resolve()

PATHS = dict(
    assets    = _ROOT / "assets" / "exploratory",
)

In [19]:
# ── 1.1 Chargement ─────────────────────────────────────────────────────────
# Adapter le chemin si nécessaire
DATA_PATH = "../../dataset/raw/handover_dataset.csv"

df_raw = pd.read_csv(DATA_PATH, comment='#')
print(f"Dataset brut : {df_raw.shape[0]:,} lignes × {df_raw.shape[1]} colonnes")
print()
print("Colonnes :")
for col in df_raw.columns:
    print(f"  {col:<35} {df_raw[col].dtype}")


Dataset brut : 90,300 lignes × 65 colonnes

Colonnes :
  timestamp                           object
  ue_id                               object
  scenario_id                         int64
  rsrp                                float64
  sinr                                float64
  rsrq                                float64
  cqi                                 int64
  position_x                          float64
  position_y                          float64
  altitude                            float64
  speed                               float64
  direction                           float64
  mobility_type                       object
  serving_cell_id                     int64
  serving_cell_type                   object
  serving_net_type                    object
  target_cell_id                      int64
  target_cell_type                    object
  handover_class                      int64
  handover_label                      object
  handover_success                    int6

In [20]:
# ── 1.2 Parsing du timestamp & tri chronologique ────────────────────────────
df_raw["timestamp"] = pd.to_datetime(
    df_raw["timestamp"],
    format="ISO8601"
)
df_raw = df_raw.sort_values(["ue_id", "timestamp"]).reset_index(drop=True)

print(f"Période : {df_raw['timestamp'].min()}  →  {df_raw['timestamp'].max()}")
print(f"UEs uniques : {df_raw['ue_id'].nunique()}")
print(f"Scénarios : {df_raw['scenario_id'].nunique()}")


Période : 2024-09-01 00:00:00  →  2024-09-01 00:01:00
UEs uniques : 300
Scénarios : 4


In [21]:
# ── 1.3 Parser les colonnes multi-valeurs (K voisins) ───────────────────────
# Ces colonnes contiennent des listes séparées par ";" (ex : nb_rsrps)
# On les convertit en listes numpy pour les analyses de voisinage.

LIST_COLS = ["nb_cell_ids", "nb_cell_types", "nb_net_types",
             "nb_rsrps", "nb_sinrs", "nb_loads", "nb_tp_ests",
             "nb_dists_m", "nb_path_losses_db", "nb_scores"]

def parse_list_col(series, dtype=float):
    """Convertit une colonne de chaînes '[a;b;c]' en liste de valeurs typées."""
    def _parse(x):
        if pd.isna(x):
            return []
        s = str(x).strip("[]").replace(",", ";")
        parts = [p.strip() for p in s.split(";") if p.strip()]
        try:
            return [dtype(p) for p in parts]
        except ValueError:
            return parts
    return series.apply(_parse)

for col in LIST_COLS:
    if col in df_raw.columns:
        dtype = str if "ids" in col or "types" in col or "net" in col or "mask" in col else float
        df_raw[f"{col}_parsed"] = parse_list_col(df_raw[col], dtype=dtype)

print("✅ Colonnes multi-valeurs parsées")
print(f"   Exemple nb_rsrps_parsed[0] : {df_raw['nb_rsrps_parsed'].iloc[0]}")


✅ Colonnes multi-valeurs parsées
   Exemple nb_rsrps_parsed[0] : [-44.0, -47.17, -75.58, -78.18, -78.95, -64.2, -81.85, -81.92]


In [22]:
# ── 1.4 Feature Engineering : variables clés ────────────────────────────────

import numpy as np
import ast

df = df_raw.copy()

# =============================================================================
# 🔹 0) CLEAN ALL PARSED LIST COLUMNS (CRITICAL FIX)
# =============================================================================

def clean_numeric_list(x):
    """Ensure x is a clean list of floats."""
    try:
        # Case 1: string like "[1, 2, 3]"
        if isinstance(x, str):
            x = ast.literal_eval(x)

        # Case 2: already list-like → convert safely
        cleaned = []
        for v in x:
            try:
                v = float(v)
                if not np.isnan(v):
                    cleaned.append(v)
            except:
                continue

        return cleaned

    except:
        return []

cols_to_clean = [
    "nb_rsrps_parsed",
    "nb_sinrs_parsed",
    "nb_scores_parsed"
]

for col in cols_to_clean:
    df[col] = df[col].apply(clean_numeric_list)

# =============================================================================
# 🔹 a) Best neighbor RSRP & hysteresis margin
# =============================================================================

def get_best_neighbor_rsrp(row):
    """Return max RSRP among neighbors excluding serving cell."""
    nb_ids   = row["nb_cell_ids_parsed"]
    nb_rsrps = row["nb_rsrps_parsed"]
    serving  = str(row["serving_cell_id"])

    candidates = [
        r for cid, r in zip(nb_ids, nb_rsrps)
        if str(cid) != serving and not np.isnan(r)
    ]

    return max(candidates) if candidates else np.nan

df["best_neighbor_rsrp"] = df.apply(get_best_neighbor_rsrp, axis=1)

# Hysteresis margin
df["hysteresis_margin"] = df["rsrp"] - df["best_neighbor_rsrp"]

# =============================================================================
# 🔹 b) Δ RSRP (temporal variation)
# =============================================================================

df["delta_rsrp"] = df.groupby("ue_id")["rsrp"].diff()
df["delta_rsrp_abs"] = df["delta_rsrp"].abs()

# =============================================================================
# 🔹 c) Serving optimal flag
# =============================================================================

df["is_serving_optimal"] = (
    df["serving_cell_id"] == df["optimal_cell_id"]
).astype(int)

# =============================================================================
# 🔹 d) Number of neighbors
# =============================================================================

df["nb_neighbors_actual"] = df["nb_rsrps_parsed"].apply(len)

# =============================================================================
# 🔹 e) RSRP gap (top-1 vs top-2)
# =============================================================================

def rsrp_gap_top2(nb_rsrps_list):
    vals = sorted(nb_rsrps_list, reverse=True)
    return (vals[0] - vals[1]) if len(vals) >= 2 else np.nan

df["rsrp_gap_top2"] = df["nb_rsrps_parsed"].apply(rsrp_gap_top2)

# =============================================================================
# 🔹 f) SINR spread (std)
# =============================================================================

def safe_std(x):
    return np.std(x) if len(x) > 1 else np.nan

df["sinr_std_neighbors"] = df["nb_sinrs_parsed"].apply(safe_std)

# =============================================================================
# 🔹 g) Score gap (ambiguity)
# =============================================================================

def score_gap_top2(x):
    vals = sorted(x, reverse=True)
    return (vals[0] - vals[1]) if len(vals) >= 2 else np.nan

df["score_gap_top2"] = df["nb_scores_parsed"].apply(score_gap_top2)

# =============================================================================
# 🔹 h) Binary label
# =============================================================================

df["is_handover"] = (df["handover_label"] != "no_handover").astype(int)

# =============================================================================
# 🔹 i) Sanity check
# =============================================================================

print("✅ Features créées :")
new_features = [
    "best_neighbor_rsrp",
    "hysteresis_margin",
    "delta_rsrp",
    "delta_rsrp_abs",
    "is_serving_optimal",
    "nb_neighbors_actual",
    "rsrp_gap_top2",
    "sinr_std_neighbors",
    "score_gap_top2",
    "is_handover"
]

for f in new_features:
    print(f"{f:<30} non-null={df[f].notna().sum():,}")

# =============================================================================
# 🔹 j) EXTRA DEBUG (HIGHLY RECOMMENDED)
# =============================================================================

print("\n🔍 Sample check:")
sample = df["nb_sinrs_parsed"].iloc[0]
print(sample)
print(type(sample))
print([type(v) for v in sample[:5]])

✅ Features créées :
best_neighbor_rsrp             non-null=89,211
hysteresis_margin              non-null=89,211
delta_rsrp                     non-null=90,000
delta_rsrp_abs                 non-null=90,000
is_serving_optimal             non-null=90,300
nb_neighbors_actual            non-null=90,300
rsrp_gap_top2                  non-null=83,740
sinr_std_neighbors             non-null=83,740
score_gap_top2                 non-null=83,740
is_handover                    non-null=90,300

🔍 Sample check:
[11.3, 5.01, 8.82, 5.18, 4.22, -13.68, 0.9, 0.81]
<class 'list'>
[<class 'float'>, <class 'float'>, <class 'float'>, <class 'float'>, <class 'float'>]


In [23]:
# ── 1.5 Aperçu général du dataset enrichi ───────────────────────────────────
print("=== Statistiques descriptives (features clés) ===")
key_cols = ["rsrp", "sinr", "speed", "cell_load", "hysteresis_margin",
            "delta_rsrp", "rsrp_gap_top2", "score_gap_top2",
            "time_to_trigger", "hysteresis"]
df[key_cols].describe().T.style.background_gradient(cmap="coolwarm", axis=1)


=== Statistiques descriptives (features clés) ===


,count,mean,std,min,25%,50%,75%,max
rsrp,90300.000000,-63.439989,13.300973,-118.590000,-72.330000,-62.690000,-52.820000,-44.000000
sinr,90300.000000,14.702882,9.801439,-39.390000,8.650000,13.890000,20.490000,60.000000
speed,90300.000000,35.109983,10.775435,1.019000,33.586750,39.919000,41.830750,43.952000
cell_load,90300.000000,0.269588,0.138009,0.054000,0.112000,0.288000,0.378000,0.563000
hysteresis_margin,89211.000000,-4.587698,12.271922,-66.060000,-10.240000,0.000000,0.000000,42.830000
delta_rsrp,90000.000000,-0.007960,6.383789,-56.410000,-3.440000,0.000000,3.150000,61.580000
rsrp_gap_top2,83740.000000,6.555043,5.871842,0.000000,1.930000,4.990000,9.720000,42.910000
score_gap_top2,83740.000000,0.098971,0.085749,0.000000,0.031200,0.077300,0.145300,0.514700
time_to_trigger,90300.000000,160.000000,0.000000,160.000000,160.000000,160.000000,160.000000,160.000000
hysteresis,90300.000000,3.000000,0.000000,3.000000,3.000000,3.000000,3.000000,3.000000


In [24]:
# ── 1.6 Valeurs manquantes ──────────────────────────────────────────────────
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
if len(missing):
    print("Colonnes avec valeurs manquantes :")
    print(missing.to_string())
else:
    print("✅ Aucune valeur manquante dans les colonnes scalaires principales.")


Colonnes avec valeurs manquantes :
rsrp_gap_top2         6560
sinr_std_neighbors    6560
score_gap_top2        6560
best_neighbor_rsrp    1089
hysteresis_margin     1089
delta_rsrp             300
delta_rsrp_abs         300


### Interprétation des résultats — Section 1
> *Documentez ici vos observations sur la qualité des données, les valeurs manquantes, et les premières anomalies détectées.*

- **Hysteresis Margin** :  
- **Delta RSRP** :  
- **Nb voisins réels** :  


---
## Section 2 — Distribution des Classes & Déséquilibre

### Pourquoi cette section est cruciale
Un modèle entraîné sur un dataset fortement déséquilibré apprend à ignorer les classes minoritaires.
Ici, `no_handover` représente la majorité écrasante des échantillons. Ce déséquilibre biaise directement
la métrique Top-1 Accuracy et explique une partie du plateau à 61%.


In [25]:
# ── 2.1 Distribution des classes ─────────────────────────────────────────────
def plot_class_distribution(df, label_col="handover_label"):
    """Visualise la distribution des classes de handover avec rappel du déséquilibre."""
    counts = df[label_col].value_counts()
    proportions = counts / counts.sum() * 100
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Distribution des Classes de Handover", fontsize=15, fontweight="bold", y=1.01)
    
    # Bar chart
    ax = axes[0]
    colors = [PALETTE_CLASS.get(c, "#888") for c in counts.index]
    bars = ax.bar(counts.index, counts.values, color=colors, edgecolor="#3d4166", linewidth=0.8)
    ax.set_title("Nombre d'échantillons par classe")
    ax.set_xlabel("Classe Handover")
    ax.set_ylabel("Count")
    ax.tick_params(axis='x', rotation=25)
    for bar, v in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                f"{v:,}\n({v/counts.sum()*100:.1f}%)", ha="center", va="bottom", fontsize=9)
    ax.grid(axis="y", alpha=0.4)
    
    # Pie
    ax = axes[1]
    wedges, texts, autotexts = ax.pie(
        counts.values, labels=counts.index, colors=colors,
        autopct="%1.1f%%", startangle=140,
        wedgeprops={"edgecolor": "#0f1117", "linewidth": 1.5},
        textprops={"fontsize": 9}
    )
    ax.set_title("Proportion des classes")
    
    plt.tight_layout()
    plt.savefig(PATHS['assets']/"fig_class_distribution.png", bbox_inches="tight", facecolor="#0f1117")
    plt.show()
    
    print("\n── Ratio de déséquilibre (max/min classe) ──")
    print(f"   Classe dominante : '{counts.idxmax()}' ({counts.max():,} échantillons)")
    print(f"   Classe rare      : '{counts.idxmin()}' ({counts.min():,} échantillons)")
    print(f"   Ratio            : {counts.max()/counts.min():.0f}:1")

plot_class_distribution(df)



── Ratio de déséquilibre (max/min classe) ──
   Classe dominante : 'no_handover' (80,698 échantillons)
   Classe rare      : 'emergency_ho' (22 échantillons)
   Ratio            : 3668:1


In [26]:
# ── 2.2 Distribution RSRP, SINR, Speed par classe ───────────────────────────
def plot_feature_by_class(df, features, label_col="handover_label"):
    """KDE des features clés par classe — révèle si les classes sont séparables."""
    n = len(features)
    fig, axes = plt.subplots(1, n, figsize=(6*n, 4))
    fig.suptitle("Distribution des Features Physiques par Classe de Handover",
                 fontsize=14, fontweight="bold")
    
    classes = df[label_col].unique()
    for ax, feat in zip(axes, features):
        for cls in classes:
            subset = df[df[label_col] == cls][feat].dropna()
            if len(subset) > 10:
                subset.plot.kde(ax=ax, label=cls,
                                color=PALETTE_CLASS.get(cls, "#aaa"),
                                linewidth=2, alpha=0.85)
        ax.set_title(feat)
        ax.set_xlabel(feat)
        ax.set_ylabel("Densité")
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(PATHS['assets']/"fig_feature_by_class.png", bbox_inches="tight", facecolor="#0f1117")
    plt.show()

plot_feature_by_class(df, ["rsrp", "sinr", "speed", "cell_load"])


In [27]:
# ── 2.3 Recall simulé par cellule (densité des handovers) ────────────────────
def plot_handover_density_per_cell(df, top_n=20):
    """
    Visualise quelles cellules génèrent le plus de handovers (target_cell_id).
    Un déséquilibre fort ici indique que certaines cellules attirent beaucoup
    de trafic (zones denses, bonnes performances) — ce que le modèle doit apprendre.
    """
    ho_df = df[df["is_handover"] == 1]
    if ho_df.empty:
        print("Aucun handover dans le dataset.")
        return
    
    cell_counts = ho_df["target_cell_id"].value_counts().head(top_n)
    
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.set_title(f"Top {top_n} Cellules Cibles de Handover — Fréquence d'Attraction",
                 fontsize=13, fontweight="bold")
    
    bars = ax.bar(range(len(cell_counts)), cell_counts.values,
                  color="#4a9eff", edgecolor="#3d4166", alpha=0.85)
    ax.set_xticks(range(len(cell_counts)))
    ax.set_xticklabels([str(c) for c in cell_counts.index], rotation=45, ha="right", fontsize=8)
    ax.set_xlabel("Cell ID")
    ax.set_ylabel("Nb de handovers reçus")
    ax.grid(axis="y", alpha=0.4)
    
    # Ligne de moyenne
    mean_val = cell_counts.mean()
    ax.axhline(mean_val, color="#ff7043", linestyle="--", linewidth=1.5,
               label=f"Moyenne = {mean_val:.0f}")
    ax.legend()
    
    plt.tight_layout()
    plt.savefig(PATHS['assets']/"fig_handover_density_cell.png", bbox_inches="tight", facecolor="#0f1117")
    plt.show()

plot_handover_density_per_cell(df)


In [28]:
# ── 2.4 Corrélation cell_load & échecs de handover ───────────────────────────
def plot_load_vs_handover(df):
    """
    Analyse si la charge cellulaire (cell_load) corrèle avec les échecs de handover.
    Une surcharge peut forcer des handovers sous-optimaux ou retarder le TTT.
    """
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle("Charge Cellulaire vs Décisions de Handover", fontsize=14, fontweight="bold")
    
    # Distribution cell_load par classe
    ax = axes[0]
    for cls, color in PALETTE_CLASS.items():
        subset = df[df["handover_label"] == cls]["cell_load"].dropna()
        if len(subset) > 5:
            subset.plot.kde(ax=ax, label=cls, color=color, linewidth=2)
    ax.set_title("Distribution cell_load par classe")
    ax.set_xlabel("Charge cellulaire")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    
    # Boxplot cell_load par classe
    ax = axes[1]
    order = df["handover_label"].value_counts().index.tolist()
    data_box = [df[df["handover_label"] == cls]["cell_load"].dropna().values for cls in order]
    bp = ax.boxplot(data_box, patch_artist=True, labels=order)
    for patch, cls in zip(bp["boxes"], order):
        patch.set_facecolor(PALETTE_CLASS.get(cls, "#888"))
        patch.set_alpha(0.7)
    ax.set_title("Boxplot cell_load par classe")
    ax.set_xlabel("Classe")
    ax.set_ylabel("cell_load")
    ax.tick_params(axis='x', rotation=25)
    ax.grid(axis="y", alpha=0.4)
    
    # Scatter cell_load vs rsrp, coloré par is_handover
    ax = axes[2]
    sc = ax.scatter(df["cell_load"], df["rsrp"],
                    c=df["is_handover"], cmap="coolwarm",
                    alpha=0.15, s=5, rasterized=True)
    plt.colorbar(sc, ax=ax, label="is_handover")
    ax.set_title("cell_load vs RSRP (couleur = handover)")
    ax.set_xlabel("cell_load")
    ax.set_ylabel("RSRP (dBm)")
    ax.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(PATHS['assets']/"fig_load_vs_handover.png", bbox_inches="tight", facecolor="#0f1117")
    plt.show()
    
    # Corrélation numérique
    corr_load_ho = df[["cell_load", "is_handover", "handover_success", "rsrp", "sinr"]].corr()
    print("\n── Corrélations avec cell_load ──")
    print(corr_load_ho["cell_load"].sort_values(ascending=False).to_string())

plot_load_vs_handover(df)



── Corrélations avec cell_load ──
cell_load           1.000000
is_handover         0.116749
handover_success    0.116749
sinr                0.021976
rsrp               -0.134399


### Interprétation des résultats — Section 2
> *Documentez ici vos observations sur le déséquilibre et la corrélation charge/handover.*

- **Ratio de déséquilibre** :  
- **Séparabilité RSRP/SINR** :  
- **Impact cell_load** :  
- **Actions recommandées** (sur-échantillonnage, class weights, focal loss...) :  


---
## Analyse du "Temporal Alias" (Time-To-Trigger)

### Pourquoi cette section est le cœur du problème
Le **Time-To-Trigger (TTT)** est le délai imposé par le standard 3GPP avant qu'un handover
soit confirmé et labellisé. Pendant ce délai, les mesures RSRP physiques peuvent déjà indiquer
qu'une cellule voisine est supérieure, mais le label restera `no_handover`.

**Ce décalage temporel crée un "alias" :** le modèle reçoit des features qui pointent vers un HO
mais un label négatif → confusion systématique → plateau de performance.

*Objectif : quantifier et visualiser ce décalage pour proposer une correction du labelling.*


In [29]:
# ── 3.1 Trajectoires RSRP individuelles avec annotation TTT ─────────────────
def plot_ue_trajectory(df, ue_id, ax=None, max_points=200):
    """
    Trace la trajectoire RSRP d'un UE dans le temps, avec :
    - RSRP de la serving cell
    - RSRP du meilleur voisin
    - Marge de hystérésis
    - Moments de changement de label
    - Paramètre TTT configuré
    """
    ue_df = df[df["ue_id"] == ue_id].copy().head(max_points)
    if ue_df.empty:
        print(f"UE {ue_id} introuvable.")
        return
    
    standalone = (ax is None)
    if standalone:
        fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
        fig.suptitle(f"Trajectoire UE : {ue_id}  |  "
                     f"Mobilité : {ue_df['mobility_type'].iloc[0]}  |  "
                     f"Vitesse moy : {ue_df['speed'].mean():.1f} m/s",
                     fontsize=13, fontweight="bold")
    else:
        axes = [ax, None, None]
    
    t = ue_df["timestamp"]
    
    # ── Sous-plot 1 : RSRP serving vs meilleur voisin ─────────────────────────
    ax0 = axes[0]
    ax0.plot(t, ue_df["rsrp"], color="#4a9eff", linewidth=2, label="RSRP Serving Cell")
    ax0.plot(t, ue_df["best_neighbor_rsrp"], color="#ff7043", linewidth=2,
             linestyle="--", label="RSRP Meilleur Voisin")
    ax0.fill_between(t, ue_df["rsrp"], ue_df["best_neighbor_rsrp"],
                     where=ue_df["rsrp"] < ue_df["best_neighbor_rsrp"],
                     alpha=0.15, color="#ff7043", label="Zone ambiguë (voisin > serving)")
    
    # Marquer les handovers
    ho_mask = ue_df["is_handover"] == 1
    if ho_mask.any():
        ax0.scatter(t[ho_mask], ue_df["rsrp"][ho_mask],
                    color="#66bb6a", s=80, zorder=5, label="Handover labelisé", marker="^")
    
    # TTT médian
    ttt_med = ue_df["time_to_trigger"].median()
    ax0.set_ylabel("RSRP (dBm)")
    ax0.set_title(f"RSRP Serving vs Meilleur Voisin  (TTT médian = {ttt_med:.0f} ms)")
    ax0.legend(fontsize=8)
    ax0.grid(alpha=0.3)
    
    if not standalone:
        return
    
    # ── Sous-plot 2 : Marge de hystérésis ─────────────────────────────────────
    ax1 = axes[1]
    ax1.plot(t, ue_df["hysteresis_margin"], color="#ab47bc", linewidth=2)
    ax1.axhline(0, color="#ffa726", linestyle="--", linewidth=1.5,
                label="Seuil 0 dB (voisin = serving)")
    ax1.fill_between(t, ue_df["hysteresis_margin"], 0,
                     where=ue_df["hysteresis_margin"] < 0,
                     alpha=0.2, color="#ff7043")
    ax1.set_ylabel("Marge (dB)")
    ax1.set_title("Marge de Hystérésis (serving - meilleur voisin) — Négatif = Candidat HO")
    ax1.legend(fontsize=8)
    ax1.grid(alpha=0.3)
    
    # ── Sous-plot 3 : Δ RSRP (vitesse de variation) ───────────────────────────
    ax2 = axes[2]
    ax2.bar(t, ue_df["delta_rsrp"], color=np.where(ue_df["delta_rsrp"] >= 0, "#4a9eff", "#ff7043"),
            alpha=0.7, width=pd.Timedelta(seconds=30))
    ax2.axhline(0, color="#e0e0f0", linewidth=0.8)
    ax2.set_ylabel("Δ RSRP (dB/step)")
    ax2.set_title("Δ RSRP — Vitesse de Variation du Signal")
    ax2.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(PATHS['assets']/f"fig_trajectory_{ue_id}.png", bbox_inches="tight", facecolor="#0f1117")
    plt.show()

# Sélection automatique de 4 UEs variés (avec HO, mobilités différentes)
ues_with_ho = df[df["is_handover"] == 1]["ue_id"].value_counts()
sample_ues = ues_with_ho.head(4).index.tolist()

print(f"UEs sélectionnés pour visualisation : {sample_ues}")
for ue in sample_ues:
    print(f"\n{'='*60}")
    print(f"UE : {ue}")
    plot_ue_trajectory(df, ue)


UEs sélectionnés pour visualisation : ['MA_UE_0130', 'MA_UE_0239', 'MA_UE_0032', 'MA_UE_0111']

UE : MA_UE_0130

UE : MA_UE_0239

UE : MA_UE_0032

UE : MA_UE_0111


In [30]:
# ── 3.2 Quantification du décalage TTT (Temporal Alias) ─────────────────────
def analyze_ttt_lag(df):
    """
    Pour chaque UE, calcule le délai entre le moment où hysteresis_margin < 0
    (voisin RSRP > serving) et le moment où le label change en handover.
    Ce décalage est l'empreinte du TTT dans les données.
    """
    lag_records = []
    
    for ue_id, ue_df in df.groupby("ue_id"):
        ue_df = ue_df.sort_values("timestamp").reset_index(drop=True)
        
        candidate_start = None
        for i, row in ue_df.iterrows():
            if row["hysteresis_margin"] < 0 and candidate_start is None:
                # Voisin devient plus fort
                candidate_start = row["timestamp"]
            elif row["is_handover"] == 1 and candidate_start is not None:
                # HO labelisé
                lag = (row["timestamp"] - candidate_start).total_seconds() * 1000  # ms
                lag_records.append({
                    "ue_id": ue_id,
                    "lag_ms": lag,
                    "ttt_config": row["time_to_trigger"],
                    "speed": row["speed"],
                    "mobility_type": row["mobility_type"],
                    "rsrp_at_ho": row["rsrp"],
                    "hysteresis_at_ho": row["hysteresis_margin"],
                })
                candidate_start = None
    
    lag_df = pd.DataFrame(lag_records)
    
    if lag_df.empty:
        print("Aucun lag TTT détecté (vérifier les données)")
        return lag_df
    
    print(f"Événements analysés : {len(lag_df)}")
    print(f"Lag moyen    : {lag_df['lag_ms'].mean():.0f} ms")
    print(f"Lag médian   : {lag_df['lag_ms'].median():.0f} ms")
    print(f"Lag max      : {lag_df['lag_ms'].max():.0f} ms")
    print(f"TTT config moy: {lag_df['ttt_config'].mean():.0f} ms")
    
    return lag_df

lag_df = analyze_ttt_lag(df)


Événements analysés : 8325
Lag moyen    : 1306 ms
Lag médian   : 600 ms
Lag max      : 59000 ms
TTT config moy: 160 ms


In [31]:
# ── 3.3 Visualisation du Temporal Alias ─────────────────────────────────────
def plot_ttt_analysis(lag_df):
    """Visualise l'écart entre le TTT configuré et le lag observé dans les données."""
    if lag_df.empty:
        print("Pas de données de lag disponibles.")
        return
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle("Analyse du Temporal Alias (Time-To-Trigger vs Lag Observé)",
                 fontsize=14, fontweight="bold")
    
    # Distribution des lags
    ax = axes[0, 0]
    lag_df["lag_ms"].hist(bins=40, ax=ax, color="#4a9eff", alpha=0.8, edgecolor="#3d4166")
    ax.axvline(lag_df["lag_ms"].mean(), color="#ff7043", linewidth=2,
               label=f"Moyenne = {lag_df['lag_ms'].mean():.0f} ms")
    ax.axvline(lag_df["ttt_config"].mean(), color="#66bb6a", linewidth=2, linestyle="--",
               label=f"TTT config = {lag_df['ttt_config'].mean():.0f} ms")
    ax.set_title("Distribution du Lag Observé vs TTT Configuré")
    ax.set_xlabel("Lag (ms)")
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    
    # Lag vs Speed
    ax = axes[0, 1]
    ax.scatter(lag_df["speed"], lag_df["lag_ms"], alpha=0.4, color="#ab47bc", s=20)
    # Tendance
    try:
        z = np.polyfit(lag_df["speed"].dropna(), lag_df["lag_ms"].dropna(), 1)
        p = np.poly1d(z)
        x_line = np.linspace(lag_df["speed"].min(), lag_df["speed"].max(), 100)
        ax.plot(x_line, p(x_line), color="#ffa726", linewidth=2, label="Tendance linéaire")
    except Exception:
        pass
    ax.set_title("Lag TTT vs Vitesse UE")
    ax.set_xlabel("Speed (m/s)")
    ax.set_ylabel("Lag (ms)")
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    
    # Lag par mobilité
    ax = axes[1, 0]
    mob_types = lag_df["mobility_type"].unique()
    data_mob = [lag_df[lag_df["mobility_type"] == m]["lag_ms"].dropna().values
                for m in mob_types]
    bp = ax.boxplot(data_mob, patch_artist=True, labels=mob_types)
    colors_mob = ["#4a9eff", "#ff7043", "#66bb6a", "#ab47bc"]
    for patch, color in zip(bp["boxes"], colors_mob):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_title("Lag TTT par Type de Mobilité")
    ax.set_ylabel("Lag (ms)")
    ax.grid(axis="y", alpha=0.4)
    
    # Corrélation Lag vs TTT config
    ax = axes[1, 1]
    ax.scatter(lag_df["ttt_config"], lag_df["lag_ms"], alpha=0.3, color="#4a9eff", s=20)
    ax.plot([lag_df["ttt_config"].min(), lag_df["ttt_config"].max()],
            [lag_df["ttt_config"].min(), lag_df["ttt_config"].max()],
            color="#ff7043", linewidth=2, linestyle="--", label="Lag = TTT (référence)")
    ax.set_title("Lag Observé vs TTT Configuré (idéalement sur la diagonale)")
    ax.set_xlabel("TTT Configuré (ms)")
    ax.set_ylabel("Lag Observé (ms)")
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(PATHS['assets']/"fig_ttt_analysis.png", bbox_inches="tight", facecolor="#0f1117")
    plt.show()

if not lag_df.empty:
    plot_ttt_analysis(lag_df)


### Interprétation des résultats — Section 3 (Temporal Alias)
> *Documentez ici l'ampleur du décalage TTT et ses implications pour le labelling.*

- **Lag moyen observé vs TTT configuré** :  
- **Influence de la vitesse sur le lag** :  
- **Mobilités les plus affectées** :  
- **Recommandation** (label anticipé, look-ahead window, correction de timestamp...) :  


---
## Section 4 — Analyse de l'Ambiguïté Décisionnelle 

In [32]:
# ── 4.1 Cas où la cible simulateur ≠ meilleur RSRP ──────────────────────────
def analyze_ambiguity(df):
    """
    Identifie les échantillons où la cellule cible du simulateur (target_cell_id)
    n'est PAS celle avec le meilleur RSRP parmi les voisins.
    → Ces cas sont ceux où le modèle Top-1 basé sur RSRP se trompe.
    """
    ho_df = df[df["is_handover"] == 1].copy()
    
    def is_target_best_rsrp(row):
        """Retourne True si la target est bien la cellule avec le meilleur RSRP."""
        nb_ids   = row["nb_cell_ids_parsed"]
        nb_rsrps = row["nb_rsrps_parsed"]
        target   = str(row["target_cell_id"])
        
        if not nb_ids or not nb_rsrps:
            return np.nan
        
        id_rsrp = {str(cid): r for cid, r in zip(nb_ids, nb_rsrps)}
        if target not in id_rsrp:
            return np.nan
        
        best_rsrp_cell = max(id_rsrp, key=id_rsrp.get)
        return best_rsrp_cell == target
    
    ho_df["target_is_best_rsrp"] = ho_df.apply(is_target_best_rsrp, axis=1)
    
    n_total   = ho_df["target_is_best_rsrp"].notna().sum()
    n_aligned = ho_df["target_is_best_rsrp"].sum()
    n_ambig   = n_total - n_aligned
    
    print("═══ Analyse d'Ambiguïté Top-1 ═══")
    print(f"Total handovers analysés     : {n_total}")
    print(f"Cible = meilleur RSRP        : {n_aligned} ({n_aligned/n_total*100:.1f}%)")
    print(f"Cible ≠ meilleur RSRP        : {n_ambig}  ({n_ambig/n_total*100:.1f}%)")
    print()
    
    return ho_df

ho_df = analyze_ambiguity(df)


═══ Analyse d'Ambiguïté Top-1 ═══
Total handovers analysés     : 9431
Cible = meilleur RSRP        : 3254 (34.5%)
Cible ≠ meilleur RSRP        : 6177  (65.5%)



In [33]:
# ── 4.3 Heatmap des corrélations (features ambiguïté) ────────────────────────
def plot_correlation_heatmap(df, subset_cols=None):
    """
    Heatmap des corrélations entre les features d'ambiguïté.
    Permet de voir quelles features sont redondantes vs complémentaires.
    """
    if subset_cols is None:
        subset_cols = [
            "rsrp", "sinr", "cell_load", "speed", "direction",
            "hysteresis_margin", "delta_rsrp", "rsrp_gap_top2",
            "score_gap_top2", "time_to_trigger", "hysteresis",
            "nb_neighbors_actual", "sinr_std_neighbors", "is_handover",
            "optimal_cell_load", "optimal_cell_score"
        ]
    available = [c for c in subset_cols if c in df.columns]
    corr = df[available].corr()
    
    fig, ax = plt.subplots(figsize=(14, 12))
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
                center=0, vmin=-1, vmax=1, ax=ax,
                annot_kws={"size": 8},
                linewidths=0.5, linecolor="#2a2d3d",
                cbar_kws={"label": "Corrélation de Pearson"})
    ax.set_title("Heatmap des Corrélations — Features d'Ambiguïté Handover",
                 fontsize=13, fontweight="bold")
    plt.xticks(rotation=40, ha="right", fontsize=9)
    plt.yticks(fontsize=9)
    plt.tight_layout()
    plt.savefig(PATHS['assets']/"fig_correlation_heatmap.png", bbox_inches="tight", facecolor="#0f1117")
    plt.show()

plot_correlation_heatmap(df)


---
## Section 5 — Analyse de la Mobilité & Phénomène Ping-Pong

In [34]:
# ── 5.1 Distribution Ping-Pong par type de mobilité ─────────────────────────
def plot_pingpong_analysis(df):
    """
    Analyse la fréquence et le contexte des ping-pong handovers.
    Un ping-pong indique une frontière de couverture instable — terrain fertile pour l'ambiguïté.
    """
    if "ping_pong_flag" not in df.columns:
        print("Colonne ping_pong_flag absente.")
        return
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle("Analyse Ping-Pong Handover", fontsize=14, fontweight="bold")
    
    # Taux de ping-pong par mobilité
    ax = axes[0]
    mob_pp = df.groupby("mobility_type")["ping_pong_flag"].agg(["sum", "count"])
    mob_pp["rate"] = mob_pp["sum"] / mob_pp["count"] * 100
    mob_pp["rate"].sort_values().plot.barh(ax=ax, color="#ff7043", alpha=0.8,
                                           edgecolor="#3d4166")
    ax.set_title("Taux de Ping-Pong par Mobilité (%)")
    ax.set_xlabel("Taux (%)")
    ax.grid(axis="x", alpha=0.4)
    
    # Ping-pong vs RSRP gap
    ax = axes[1]
    pp_rsrp  = df[df["ping_pong_flag"] == 1]["rsrp_gap_top2"].dropna()
    npp_rsrp = df[df["ping_pong_flag"] == 0]["rsrp_gap_top2"].dropna()
    if len(pp_rsrp) > 3 and len(npp_rsrp) > 3:
        pp_rsrp.plot.kde(ax=ax, color="#ff7043", linewidth=2, label=f"Ping-Pong (n={len(pp_rsrp)})")
        npp_rsrp.plot.kde(ax=ax, color="#4a9eff", linewidth=2, label=f"Normal (n={len(npp_rsrp)})")
    ax.set_title("Δ RSRP Top2 : Ping-Pong vs Normal\n(faible gap = zone ambiguë)")
    ax.set_xlabel("RSRP Gap Top2 (dB)")
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    
    # Ping-pong vs Speed
    ax = axes[2]
    speed_bins = pd.cut(df["speed"], bins=[0, 5, 20, 60, 200],
                        labels=["Piéton", "Lent", "Rapide", "Autoroute"])
    pp_by_speed = df.groupby(speed_bins, observed=True)["ping_pong_flag"].mean() * 100
    pp_by_speed.plot.bar(ax=ax, color="#ab47bc", alpha=0.8, edgecolor="#3d4166")
    ax.set_title("Taux Ping-Pong par Plage de Vitesse")
    ax.set_xlabel("Vitesse")
    ax.set_ylabel("Taux (%)")
    ax.tick_params(axis='x', rotation=0)
    ax.grid(axis="y", alpha=0.4)
    
    plt.tight_layout()
    plt.savefig(PATHS['assets']/"fig_pingpong.png", bbox_inches="tight", facecolor="#0f1117")
    plt.show()
    
    print(f"\nTaux de ping-pong global : {df['ping_pong_flag'].mean()*100:.2f}%")
    print(df.groupby("mobility_type")["ping_pong_flag"].mean().sort_values(ascending=False)
           .apply(lambda x: f"{x*100:.2f}%").to_string())

plot_pingpong_analysis(df)



Taux de ping-pong global : 5.30%
mobility_type
high_speed         5.86%
highway_vehicle    5.69%
urban_vehicle      4.85%
pedestrian         0.87%


In [35]:
# ── 5.2 Impact vitesse & direction sur la fréquence HO ──────────────────────
def plot_mobility_impact(df):
    """
    Analyse bidimensionnelle (vitesse × direction) de la fréquence de handover.
    Révèle si certains corridors de direction ont plus d'ambiguïté.
    """
    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    fig.suptitle("Impact de la Mobilité sur les Décisions de Handover",
                 fontsize=14, fontweight="bold")
    
    # Handover rate vs Speed (continue)
    ax = axes[0]
    speed_bins = pd.cut(df["speed"], bins=15)
    ho_by_speed = df.groupby(speed_bins, observed=True)["is_handover"].mean() * 100
    x_vals = [iv.mid for iv in ho_by_speed.index]
    ax.plot(x_vals, ho_by_speed.values, color="#4a9eff", linewidth=2.5, marker="o", markersize=5)
    ax.fill_between(x_vals, ho_by_speed.values, alpha=0.2, color="#4a9eff")
    ax.set_title("Taux de HO vs Vitesse UE")
    ax.set_xlabel("Vitesse (m/s)")
    ax.set_ylabel("Taux handover (%)")
    ax.grid(alpha=0.4)
    
    # Polar plot : direction vs taux HO
    ax = axes[1]
    ax.remove()
    ax_polar = fig.add_subplot(1, 3, 2, projection="polar", facecolor="#1a1d27")
    ax_polar.set_facecolor("#1a1d27")
    n_bins = 16
    dir_bins = pd.cut(df["direction"] % 360, bins=n_bins,
                      labels=np.linspace(0, 360, n_bins, endpoint=False))
    ho_by_dir = df.groupby(dir_bins, observed=True)["is_handover"].mean()
    angles = np.linspace(0, 2 * np.pi, n_bins, endpoint=False)
    vals = ho_by_dir.values
    vals_closed = np.append(vals, vals[0])
    angles_closed = np.append(angles, angles[0])
    ax_polar.plot(angles_closed, vals_closed, color="#ff7043", linewidth=2)
    ax_polar.fill(angles_closed, vals_closed, alpha=0.2, color="#ff7043")
    ax_polar.set_title("Taux HO par Direction (N=0°)", pad=15)
    ax_polar.tick_params(colors="#a0a0c0")
    
    # Scatter speed × delta_rsrp coloré par HO
    ax = axes[2]
    sc = ax.scatter(df["speed"], df["delta_rsrp"].clip(-10, 10),
                    c=df["is_handover"], cmap="RdBu_r",
                    alpha=0.15, s=4, rasterized=True)
    plt.colorbar(sc, ax=ax, label="is_handover")
    ax.set_title("Vitesse vs Δ RSRP (couleur = handover)")
    ax.set_xlabel("Speed (m/s)")
    ax.set_ylabel("Δ RSRP (dB/step)")
    ax.axhline(0, color="#ffa726", linewidth=1, linestyle="--")
    ax.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(PATHS['assets']/"fig_mobility_impact.png", bbox_inches="tight", facecolor="#0f1117")
    plt.show()

plot_mobility_impact(df)


In [36]:
# ── 5.3 Fréquence HO par UE (détection des UEs instables) ──────────────────
def plot_ue_instability(df, top_n=25):
    """
    Identifie les UEs qui font le plus de handovers.
    Ces UEs sont les plus exposés à l'ambiguïté et aux ping-pongs.
    """
    ue_ho_rate = df.groupby("ue_id").agg(
        total_steps=("timestamp", "count"),
        n_ho=("is_handover", "sum"),
        n_pp=("ping_pong_flag", "sum"),
        mean_speed=("speed", "mean"),
        mobility=("mobility_type", lambda x: x.mode()[0] if len(x) > 0 else "unknown")
    )
    ue_ho_rate["ho_rate"] = ue_ho_rate["n_ho"] / ue_ho_rate["total_steps"] * 100
    ue_ho_rate = ue_ho_rate.sort_values("ho_rate", ascending=False).head(top_n)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f"Top {top_n} UEs les Plus Instables (taux HO élevé)",
                 fontsize=13, fontweight="bold")
    
    # Bar du taux HO
    ax = axes[0]
    mob_colors = {"highway": "#4a9eff", "urban_vehicle": "#ff7043",
                  "pedestrian": "#66bb6a", "stationary": "#ab47bc"}
    colors = [mob_colors.get(m, "#888") for m in ue_ho_rate["mobility"]]
    bars = ax.barh(range(len(ue_ho_rate)), ue_ho_rate["ho_rate"].values,
                   color=colors, alpha=0.8, edgecolor="#3d4166")
    ax.set_yticks(range(len(ue_ho_rate)))
    ax.set_yticklabels(ue_ho_rate.index, fontsize=7)
    ax.set_xlabel("Taux de Handover (%)")
    ax.set_title("Taux HO par UE")
    ax.grid(axis="x", alpha=0.4)
    
    # Scatter HO rate vs speed
    ax = axes[1]
    sc = ax.scatter(ue_ho_rate["mean_speed"], ue_ho_rate["ho_rate"],
                    c=ue_ho_rate["n_pp"], cmap="YlOrRd", s=60, alpha=0.8,
                    edgecolors="#3d4166", linewidth=0.5)
    plt.colorbar(sc, ax=ax, label="Nb Ping-Pong")
    ax.set_xlabel("Vitesse moyenne (m/s)")
    ax.set_ylabel("Taux HO (%)")
    ax.set_title("Instabilité : Speed vs HO Rate (couleur = ping-pong count)")
    ax.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(PATHS['assets']/"fig_ue_instability.png", bbox_inches="tight", facecolor="#0f1117")
    plt.show()

plot_ue_instability(df)


### Interprétation des résultats — Section 5 (Mobilité)
> *Documentez ici les patterns de mobilité et leur lien avec l'instabilité.*

- **Taux de ping-pong global** :  
- **Mobilité la plus affectée** :  
- **Direction dominante pour les HO** :  
- **Seuil de vitesse critique** :  
- **Recommandation** (features de mobilité à intégrer au modèle...) :  


---
## Section 6 — Analyse de l'Espace des Voisins (K-Neighbors)

### Pourquoi cette section est nouvelle pour le modèle
Le modèle DeepSet/Transformer reçoit en entrée les K voisins. Cette section analyse
**la distribution intrinsèque de cet espace** : combien de voisins sont réellement compétitifs ?
Quelle est la densité typique de la zone ambiguë ? Comment les scores composites se comportent ?


In [37]:
# ── 6.1 Distribution du nombre de voisins compétitifs ───────────────────────
def analyze_neighbor_space(df, rsrp_threshold_db=3.0):
    """
    Calcule, pour chaque UE/timestamp, le nombre de voisins 'compétitifs'
    (dans un seuil de rsrp_threshold_db dB du meilleur voisin).
    Un grand nombre de voisins compétitifs = forte ambiguïté.
    """
    def count_competitive(nb_rsrps_list, threshold=rsrp_threshold_db):
        if not nb_rsrps_list:
            return 0
        best = max(nb_rsrps_list)
        return sum(1 for r in nb_rsrps_list if (best - r) <= threshold)
    
    df["n_competitive_neighbors"] = df["nb_rsrps_parsed"].apply(count_competitive)
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle(f"Espace des Voisins Compétitifs (seuil = {rsrp_threshold_db} dB)",
                 fontsize=14, fontweight="bold")
    
    # Distribution
    ax = axes[0]
    df["n_competitive_neighbors"].value_counts().sort_index().plot.bar(
        ax=ax, color="#4a9eff", alpha=0.8, edgecolor="#3d4166")
    ax.set_title("Distribution du nb de voisins compétitifs")
    ax.set_xlabel("Nb de voisins compétitifs")
    ax.set_ylabel("Fréquence")
    ax.grid(axis="y", alpha=0.4)
    
    # Compétition vs classe HO
    ax = axes[1]
    for cls, color in PALETTE_CLASS.items():
        subset = df[df["handover_label"] == cls]["n_competitive_neighbors"]
        if len(subset) > 5:
            subset.value_counts().sort_index().plot(ax=ax, label=cls, color=color,
                                                    linewidth=2, alpha=0.85)
    ax.set_title("Voisins compétitifs par classe de HO")
    ax.set_xlabel("Nb voisins compétitifs")
    ax.set_ylabel("Fréquence")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    
    # Compétition vs score_gap
    ax = axes[2]
    sc = ax.scatter(df["n_competitive_neighbors"], df["score_gap_top2"],
                    c=df["is_handover"], cmap="coolwarm",
                    alpha=0.2, s=6, rasterized=True)
    plt.colorbar(sc, ax=ax, label="is_handover")
    ax.set_title("Nb voisins compétitifs vs Score Gap Top2")
    ax.set_xlabel("Nb voisins compétitifs")
    ax.set_ylabel("Score gap top-2")
    ax.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig("fig_neighbor_space.png", bbox_inches="tight", facecolor="#0f1117")
    plt.show()
    
    print(f"\n── Nb moyen de voisins compétitifs ──")
    print(df.groupby("handover_label")["n_competitive_neighbors"].describe().round(2).to_string())

analyze_neighbor_space(df)



── Nb moyen de voisins compétitifs ──
                     count  mean   std  min  25%  50%  75%  max
handover_label                                                 
drone_to_macro_ho    781.0  1.55  0.78  1.0  1.0  1.0  2.0  5.0
emergency_ho          22.0  1.14  0.35  1.0  1.0  1.0  1.0  2.0
intra_freq_ho       8037.0  1.46  0.71  1.0  1.0  1.0  2.0  7.0
macro_to_drone_ho    762.0  1.60  0.82  1.0  1.0  1.0  2.0  5.0
no_handover        80698.0  1.42  0.69  1.0  1.0  1.0  2.0  7.0


In [38]:
# ── 6.2 Distribution des scores composites des voisins ──────────────────────
def plot_score_distribution(df):
    """
    Analyse la distribution des scores composites (nb_scores) qui pilotent la décision.
    Un faible écart entre scores = ambiguïté directe pour le modèle.
    """
    # Extraire tous les scores de voisins (aplatir)
    all_scores = []
    for scores_list in df["nb_scores_parsed"].dropna():
        all_scores.extend(scores_list)
    all_scores = np.array(all_scores)
    
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle("Distribution des Scores Composites des Voisins", fontsize=13, fontweight="bold")
    
    ax = axes[0]
    ax.hist(all_scores, bins=60, color="#66bb6a", alpha=0.8, edgecolor="#3d4166", density=True)
    ax.set_title("Distribution globale des scores")
    ax.set_xlabel("Score composite")
    ax.set_ylabel("Densité")
    ax.grid(alpha=0.3)
    
    ax = axes[1]
    score_gap_by_class = df.groupby("handover_label")["score_gap_top2"].apply(
        lambda x: x.dropna().values)
    data = [v for v in score_gap_by_class.values if len(v) > 0]
    labels = [k for k, v in score_gap_by_class.items() if len(v) > 0]
    bp = ax.boxplot(data, patch_artist=True, labels=labels)
    for patch, cls in zip(bp["boxes"], labels):
        patch.set_facecolor(PALETTE_CLASS.get(cls, "#888"))
        patch.set_alpha(0.7)
    ax.set_title("Score Gap Top-2 par Classe de HO")
    ax.set_xlabel("Classe")
    ax.set_ylabel("Score gap top-2 (plus faible = plus ambigu)")
    ax.tick_params(axis='x', rotation=20)
    ax.grid(axis="y", alpha=0.4)
    
    plt.tight_layout()
    plt.savefig(PATHS['assets']/"fig_score_distribution.png", bbox_inches="tight", facecolor="#0f1117")
    plt.show()

plot_score_distribution(df)


###  Interprétation des résultats — Section 6 (Voisins)
> *Documentez ici la structure de l'espace des K voisins.*

- **Nb moyen de voisins compétitifs** :  
- **Seuil de score_gap en dessous duquel l'ambiguïté explose** :  
- **Recommandation** (seuil adaptatif, features d'entropie du voisinage...) :  


---
## Section 7 — Dashboard de Synthèse & Recommandations

Cette section consolide les métriques clés en un seul tableau de bord décisionnel.


In [39]:
# ── 7.1 Dashboard synthèse ──────────────────────────────────────────────────
def plot_summary_dashboard(df, lag_df=None):
    """Dashboard de synthèse multi-panneaux pour présentation stakeholder."""
    
    fig = plt.figure(figsize=(20, 14))
    fig.patch.set_facecolor("#0f1117")
    gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.35)
    
    # ── 1. Classes (pie) ─────────────────────────────────────────────────────
    ax1 = fig.add_subplot(gs[0, 0])
    counts = df["handover_label"].value_counts()
    colors = [PALETTE_CLASS.get(c, "#888") for c in counts.index]
    ax1.pie(counts.values, labels=[c.replace("_", "") for c in counts.index],
            colors=colors, autopct="%1.0f%%", startangle=140,
            textprops={"fontsize": 7},
            wedgeprops={"edgecolor": "#0f1117", "linewidth": 1.2})
    ax1.set_title("Classes HO", fontsize=11)
    
    # ── 2. RSRP par classe ────────────────────────────────────────────────────
    ax2 = fig.add_subplot(gs[0, 1])
    for cls, color in PALETTE_CLASS.items():
        s = df[df["handover_label"] == cls]["rsrp"].dropna()
        if len(s) > 10:
            s.plot.kde(ax=ax2, color=color, linewidth=2, label=cls.replace("_", " "), alpha=0.85)
    ax2.set_title("RSRP par Classe", fontsize=11)
    ax2.set_xlabel("RSRP (dBm)", fontsize=9)
    ax2.legend(fontsize=6)
    ax2.grid(alpha=0.3)
    
    # ── 3. Hysteresis margin ──────────────────────────────────────────────────
    ax3 = fig.add_subplot(gs[0, 2])
    df["hysteresis_margin"].clip(-30, 30).hist(bins=50, ax=ax3, color="#ab47bc",
                                                alpha=0.8, edgecolor="#3d4166")
    ax3.axvline(0, color="#ff7043", linestyle="--", linewidth=2,
                label="Seuil HO (0 dB)")
    ax3.set_title("Marge de Hystérésis", fontsize=11)
    ax3.set_xlabel("dB", fontsize=9)
    ax3.legend(fontsize=8)
    ax3.grid(alpha=0.3)
    
    # ── 4. Score gap top-2 ────────────────────────────────────────────────────
    ax4 = fig.add_subplot(gs[0, 3])
    df["score_gap_top2"].clip(0, 0.5).hist(bins=40, ax=ax4, color="#ffa726",
                                            alpha=0.8, edgecolor="#3d4166")
    ax4.axvline(df["score_gap_top2"].median(), color="#4a9eff", linewidth=2,
                linestyle="--", label=f"Médiane = {df['score_gap_top2'].median():.3f}")
    ax4.set_title("Ambiguïté : Score Gap Top-2", fontsize=11)
    ax4.set_xlabel("Score gap", fontsize=9)
    ax4.legend(fontsize=8)
    ax4.grid(alpha=0.3)
    
    # ── 5. Ping-Pong par mobilité ─────────────────────────────────────────────
    ax5 = fig.add_subplot(gs[1, :2])
    pp_mob = df.groupby("mobility_type")["ping_pong_flag"].mean() * 100
    colors5 = ["#4a9eff", "#ff7043", "#66bb6a", "#ab47bc"]
    pp_mob.sort_values().plot.barh(ax=ax5, color=colors5[:len(pp_mob)],
                                   alpha=0.8, edgecolor="#3d4166")
    ax5.set_title("Taux Ping-Pong par Mobilité", fontsize=11)
    ax5.set_xlabel("Taux (%)")
    ax5.grid(axis="x", alpha=0.4)
    
    # ── 6. TTT Lag ────────────────────────────────────────────────────────────
    ax6 = fig.add_subplot(gs[1, 2:])
    if lag_df is not None and not lag_df.empty:
        lag_df["lag_ms"].clip(0, 2000).hist(bins=40, ax=ax6, color="#4a9eff",
                                             alpha=0.8, edgecolor="#3d4166")
        ax6.axvline(lag_df["lag_ms"].mean(), color="#ff7043", linewidth=2, linestyle="--",
                    label=f"Lag moyen = {lag_df['lag_ms'].mean():.0f} ms")
        if "ttt_config" in lag_df.columns:
            ax6.axvline(lag_df["ttt_config"].mean(), color="#66bb6a", linewidth=2, linestyle=":",
                        label=f"TTT config = {lag_df['ttt_config'].mean():.0f} ms")
        ax6.legend(fontsize=8)
    else:
        ax6.text(0.5, 0.5, "Données TTT lag\nnon disponibles",
                 ha="center", va="center", transform=ax6.transAxes)
    ax6.set_title("Distribution Lag TTT (Temporal Alias)", fontsize=11)
    ax6.set_xlabel("Lag (ms)")
    ax6.grid(alpha=0.3)
    
    # ── 7. Table métriques clés ───────────────────────────────────────────────
    ax7 = fig.add_subplot(gs[2, :])
    ax7.axis("off")
    
    metrics = [
        ["Métrique", "Valeur", "Interprétation"],
        ["Total échantillons", f"{len(df):,}", "Volume dataset"],
        ["Taux de HO global", f"{df['is_handover'].mean()*100:.1f}%", "Déséquilibre classes"],
        ["Taux ping-pong", f"{df['ping_pong_flag'].mean()*100:.2f}%", "Instabilité décisions"],
        ["Marge hystérésis médiane", f"{df['hysteresis_margin'].median():.1f} dB", "Sécurité serving cell"],
        ["Score gap top-2 médian", f"{df['score_gap_top2'].median():.3f}", "Niveau d'ambiguïté"],
        ["Nb voisins moyen", f"{df['nb_neighbors_actual'].mean():.1f}", "Densité réseau"],
        ["UEs uniques", f"{df['ue_id'].nunique()}", "Diversité mobilité"],
        ["TTT configuré médian", f"{df['time_to_trigger'].median():.0f} ms", "Délai labelisation"],
        ["Lag TTT moyen observé",
         f"{lag_df['lag_ms'].mean():.0f} ms" if lag_df is not None and not lag_df.empty else "N/A",
         "Décalage temporel réel"],
    ]
    
    table = ax7.table(cellText=metrics[1:], colLabels=metrics[0],
                      cellLoc="center", loc="center",
                      bbox=[0, 0, 1, 1])
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    for (row, col), cell in table.get_celld().items():
        cell.set_facecolor("#1a1d27" if row % 2 == 0 else "#22253a")
        cell.set_edgecolor("#3d4166")
        cell.set_text_props(color="#e0e0f0")
        if row == 0:
            cell.set_facecolor("#2a2d4d")
            cell.set_text_props(color="#c8c8f0", fontweight="bold")
    
    fig.suptitle("🛰️ Dashboard EDA — Handover 6G  |  Diagnostic Plateau 61% Top-1 Accuracy",
                 fontsize=15, fontweight="bold", y=1.01)
    
    plt.savefig(PATHS['assets']/"fig_dashboard_synthese.png", bbox_inches="tight", facecolor="#0f1117", dpi=150)
    plt.show()
    print("✅ Dashboard sauvegardé.")

try:
    plot_summary_dashboard(df, lag_df)
except Exception as e:
    print(f"Dashboard partiel : {e}")
    plot_summary_dashboard(df)


✅ Dashboard sauvegardé.


---
## Section 8 — Recommandations pour Débloquer le Plateau

### Synthèse des causes identifiées

| Cause | Symptôme | Solution |
|-------|----------|----------|
| **Temporal Alias (TTT)** | Label décalé vs réalité physique | Labelling look-ahead + correction de fenêtre |
| **Ambiguïté de voisinage** | Plusieurs cellules dans ±3 dB | Feature d'entropie RSRP, focal loss |
| **Déséquilibre classes** | 90%+ no_handover | SMOTE, class weights, sous-échantillonnage |
| **Critères cachés simulateur** | Cible ≠ meilleur RSRP | Intégrer cell_load, distance, type réseau |
| **Ping-pong** | Frontière instable | Hysteresis adaptative, mémoire temporelle |

### Nouvelles features recommandées
```python
# Features d'ambiguïté
"rsrp_gap_top2"           # Écart RSRP 1er-2ème voisin
"score_gap_top2"          # Écart score composite  
"n_competitive_neighbors" # Nb voisins dans ±3 dB
"sinr_std_neighbors"      # Variance SINR voisinage
"hysteresis_margin"       # Marge serving - meilleur voisin

# Features temporelles (anti-TTT alias)
"delta_rsrp"              # Vitesse de variation RSRP
"delta_rsrp_abs"          # Intensité de la variation

# Features contextuelles
"interference_level"      # Niveau d'interférence
"rlf_flag"                # Risque de RLF (Radio Link Failure)
```

### 📝 Plan d'action
> *Documentez ici votre plan d'action prioritaire.*

1. **Court terme** :  
2. **Moyen terme** :  
3. **Long terme** :  


---
## Distribution du nombre de cellules détectées
Analyse de la distribution du nombre de cellules détectées par équipement (basée sur la longueur de `nb_cell_ids`).

In [40]:
# Distribution du nombre de cellules voisines détectées (nb_cell_ids)
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Vérifier si nb_cell_ids_parsed existe, sinon utiliser nb_neighbors_actual
if 'nb_cell_ids_parsed' in df.columns:
    num_cells = df['nb_cell_ids_parsed'].apply(len)
elif 'nb_neighbors_actual' in df.columns:
    num_cells = df['nb_neighbors_actual']
else:
    # Fallback au cas où le feature engineering n'est pas encore exécuté
    num_cells = df['nb_cell_ids'].apply(lambda x: len(str(x).strip("[]").split(";")) if pd.notna(x) else 0)

plt.figure(figsize=(10, 6))
ax = sns.countplot(x=num_cells, palette='viridis')
plt.title('Distribution du nombre de cellules détectées par échantillon', pad=15)
plt.xlabel('Nombre de cellules (nb_cell_ids)', labelpad=10)
plt.ylabel('Fréquence', labelpad=10)

# Ajouter les pourcentages au-dessus des barres
total = len(num_cells)
for p in ax.patches:
    height = p.get_height()
    if height > 0:
        percentage = f'{100 * height / total:.1f}%'
        x = p.get_x() + p.get_width() / 2
        y = height
        ax.annotate(percentage, (x, y), ha='center', va='bottom', fontsize=10)

plt.savefig(PATHS['assets']/"fig_frequence_detected_cell.png", bbox_inches="tight", facecolor="#0f1117", dpi=150)
plt.show()


# Afficher les statistiques de base
print("Statistiques descriptives du nombre de cellules détectées :")
display(num_cells.describe())


Statistiques descriptives du nombre de cellules détectées :


count    90300.000000
mean         6.804363
std          2.188727
min          1.000000
25%          6.000000
50%          8.000000
75%          8.000000
max          8.000000
Name: nb_cell_ids_parsed, dtype: float64